In [7]:
# 최신 Langchain 및 관련 패키지
!pip install -qU langchain-core langchain-openai langchain-community \
langchain-text-splitters faiss-cpu tiktoken


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get("OPENAI_API_KEY")

-  프롬프트 (Prompt) 정의
사용자와 언어 모델(LLM) 간의 상호작용에서 모델의 응답 품질과 방향을 결정짓는 핵심 입력값입니다.

- 프롬프트 작성 6대 원칙
1. 명확성과 구체성: 모호함을 제거하고, 모델이 이해할 수 있는 명확한 지시어를 사용합니다.

2. 배경 정보 제공: 풍부한 컨텍스트를 제공하여 환각(Hallucination) 현상을 방지하고 정확도를 높입니다.

3. 간결함 유지: 불필요한 정보는 배제하고 핵심적인 작업에만 집중합니다.

4. 열린 질문 활용: "예/아니오" 단답형을 넘어, 모델이 상세하고 분석적인 답변을 생성하도록 유도합니다.

5. 명확한 목표 설정: 기대하는 결과물의 형식(표, 코드, 요약 등)과 범위를 사전에 정의합니다.

6. 맥락에 맞는 언어/문체: 대화의 목적과 대상 독자에 적합한 톤앤매너를 설정합니다.

- 프롬프트 템플릿 (Prompt Template) 구조
효율적인 자동화 및 재사용성을 위해 프롬프트를 구성하는 4가지 필수 요소입니다. 최신 LangChain에서는 ChatPromptTemplate을 통해 이를 구조화합니다.

- 프롬프트 템플릿의 구성 요소
1. 지시 (Instruction): 모델이 수행해야 할 구체적인 작업(예: "너는 파이썬 전문가로서 코드를 리뷰해줘.")

2. 예시 (Few-shot): 작업 방식에 대한 모범 답안을 제공하여 모델의 이해도를 극대화합니다.

3. 맥락 (Context): 작업을 수행하는 데 필요한 추가 지식이나 데이터(RAG의 핵심).

4. 질문 (Input/Question): 모델이 답변해야 할 구체적인 대상.

In [4]:
# legacy 방식
# 문자열 템플릿
from langchain_core.prompts import PromptTemplate

# name과 age라는 두개의 변수를 사용하여 문자열 프롬픝 템플릿을 정의
tmp_txt = '안녕하세요. 내 이를은 {name}이고, 나이는 {age}살 입니다'

prompt_tmp = PromptTemplate.from_template(tmp_txt)

# format()를 사용하여 템플릿의 변수에 값을 채워 완전한 프롬프트 문자열을 생성
filled_prompt = prompt_tmp.format(name='홍길동', age=20)

filled_prompt

'안녕하세요. 내 이를은 홍길동이고, 나이는 20살 입니다'

In [8]:
# 최신 표준 리팩토링 코드

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
import os
from google.colab import userdata

# 1. API 키 설정
os.environ['OPENAI_API_KEY'] = userdata.get("OPENAI_API_KEY")

# 2. 모델 설정
model = ChatOpenAI(model = "gpt-4o-mini", temperature= 0.7)

# 3. 최신 방식 :  ChatPromptTemplate 정의
# 이전의 단순 문자열 템플릿보다 역할(role)을 명확히 정의 한다
prompt_tmplate = ChatPromptTemplate.from_template(
    "자기 소개를 하는 문장을 완성해 주세요. 이름 : {name}, 나이 : {age} 살"
)

# 4. LCEL 파이프라인 구성
# prompt | model | StrOutputParser 순
chain = prompt_tmplate | model | StrOutputParser()

# 5. 실행 (invoke를 통한 딕셔너리 전달)
result = chain.invoke({"name": "홍길동", "age": 20})

print(result)

안녕하세요, 저는 홍길동이고, 20살입니다. 만나서 반갑습니다!


In [10]:
# legacy
# 프롬프트 템플릿간 결합
# -> 문자열 + 문자열
# -> PromptTemplate + PromptTemplate
# -> PromptTemplate + 문자열

# -> PromptTemplate + PromptTemplate + 문자열

combined_prompt = (
    prompt_tmp
    + PromptTemplate.from_template("\n\n아버지를 아버지라 부르지 못한다")
    + '\n\n{language}로 번역해줘.'
)

combined_prompt

print(combined_prompt.format(name='홍길동', age=20, language='영어'))

안녕하세요. 내 이를은 홍길동이고, 나이는 20살 입니다

아버지를 아버지라 부르지 못한다

영어로 번역해줘.


In [13]:
# 최신 표준 리팩토링 코드

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
import os
from google.colab import userdata

# 1. API 키 설정
os.environ['OPENAI_API_KEY'] = userdata.get("OPENAI_API_KEY")

# 2. 모델 설정
model = ChatOpenAI(model = "gpt-4o-mini", temperature= 0.7)

# 3. 최신 방식 :  ChatPromptTemplate의 리스트 결합및 구성
# 각 부분을 메시지 구성요소로 나누어 관리하면 가독성과 유지보수가 좋아진다
combined_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "안녕하세요. 내 이름은 {name}이고, 나이는 {age}살 입니다."),
        ("human", "아버지를 아버지라 부르지 못한다."),
        ("human", "{language}로 번역해줘.")
    ]
)

# 4. LCEL 파이프라인 구성
# prompt | model | StrOutputParser 순
chain = combined_prompt | model | StrOutputParser()

# 5. 실행 (invoke를 통한 딕셔너리 전달)
result = chain.invoke({"name": "홍길동", "age": 20, "language": "영어"})

print(result)

"I cannot call my father 'father'."


챗 프롬프트 템플릿
  - 대화형 상황에서 여러 메시지 기반으로 단일 메시지 응답을 생성하는 데 사용한다
  - 대화형 모델이나 챗봇에 주로 사용한다
  - 입력은 여러 메시지를 원소로 갖는 리스트로 구성되고, 각 메시지는 role(역할)과 내용(content)로 구성된다
  - Message 유형
    1. SystemMessage : 시스템의 기능을 설명
    2. HumanMessage : 사용자 질문을 나타낸다
    3. AIMessage : AI 모델의 응답을 제공
    4. FunctionMessage : 특정 함수 호출의 결과
    5. ToolMessage : 도구 호출의 결

- 리팩토링 요약
1. API 교체: 더 이상 지원되지 않는 assist 모듈을 완전히 제거하고, 최신 LangChain에서 권장하는 Prompt | Model | Parser 구조인 LCEL로 표준화했습니다.

2. 실행 최적화: chain.invoke() 메서드를 사용하여 입력 데이터를 딕셔너리 형태로 전달하며, StrOutputParser()를 도입해 출력 결과가 AI 객체가 아닌 깔끔한 마크다운 텍스트로 바로 나오도록 수정했습니다.

3. 모델 고도화: 문서 작성의 품질을 높이기 위해 모델을 gpt-4o로 지정하였습니다.

assist() 대신 체인 생성 (standard runnable composition)

LLMChain + PromptTemplate 조합

In [14]:
# 최신 표준 리팩토링 코드

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
import os
from google.colab import userdata

# 1. API 키 설정
os.environ['OPENAI_API_KEY'] = userdata.get("OPENAI_API_KEY")

# 2. 모델 설정
model = ChatOpenAI(temperature= 0.7)

# 3. 최신 방식 :  ChatPromptTemplate의 리스트 결합및 구성
# 역할 기반 프롬프트 정의
prompt = ChatPromptTemplate.from_messages(
    [
        ("system",
         "너는 숙련된 기술 문서 작성자야."
         "개발자들을 위한 문서를 마크다운 형식으로 작성해."
         "구조는 다음과 같아 : \n\n"
         "## 개요\n\n"
         "## 핵심 개념\n\n"
         "## 파이썬 코드 예제\n\n"
         "## 참고 자료\n\n"
         "항상 한국어로 작성해."),
        ("human", "{topic}")
    ]
)

# 4. LCEL 파이프라인 구성
# prompt | model
tech_chain = prompt | model


topic = "Langchain의 LLMChain 구성 요소 설명해줘"

# 5. 실행 (invoke를 통한 딕셔너리 전달)
result = tech_chain.invoke({"topic": topic})

print(result.content)

## 개요
LLMChain은 Langchain에서 사용되는 특별한 구성 요소 중 하나로, 언어 모델(Language Model)과 체인(Chain)을 결합한 용어입니다. 이는 Langchain의 핵심적인 부분으로, 다양한 자연어 처리 및 기계 학습 작업에 활용됩니다.

## 핵심 개념
LLMChain은 주어진 문제에 대해 지도 및 비지도 학습을 수행하는 데 사용됩니다. 이는 주어진 입력에 대한 다음 단어나 문장을 생성하거나 분류하는 작업에 적합합니다. LLMChain은 입력 데이터에 대한 확률 분포를 학습하고, 이를 바탕으로 다음 단어 또는 문장을 예측합니다.

## 파이썬 코드 예제
```python
from langchain import LLMChain

# LLMChain 모델 초기화
llm_model = LLMChain()

# 데이터 학습
llm_model.train(data)

# 다음 단어 예측
next_word = llm_model.predict_next_word(input_text)

# 다음 문장 생성
generated_sentence = llm_model.generate_sentence(start_text)
```

## 참고 자료
- Langchain 공식 문서: [https://langchain-docs.com](https://langchain-docs.com)
- 자연어 처리와 언어 모델에 대한 자세한 정보: "Natural Language Processing in Python" by David Talbot


In [16]:
# 최신 표준 리팩토링 코드

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
import os
from google.colab import userdata

# 1. API 키 설정
os.environ['OPENAI_API_KEY'] = userdata.get("OPENAI_API_KEY")

# 2. 모델 설정
model = ChatOpenAI(model = "gpt-4o", temperature= 0.7)

# 3. 최신 방식 :  ChatPromptTemplate의 리스트 결합및 구성
# 역할 기반 프롬프트 정의
prompt = ChatPromptTemplate.from_messages(
    [
        ("system",
         "너는 숙련된 기술 문서 작성자야."
         "개발자들을 위한 문서를 마크다운 형식으로 작성해."
         "구조는 다음과 같아 : \n\n"
         "## 개요\n\n"
         "## 핵심 개념\n\n"
         "## 파이썬 코드 예제\n\n"
         "## 참고 자료\n\n"
         "항상 한국어로 작성해."),
        ("human", "{topic}")
    ]
)

# 4. LCEL 파이프라인 구성
# prompt | model
tech_chain = prompt | model | StrOutputParser()


topic = "Langchain의 LLMChain 구성 요소 설명해줘"

# 5. 실행 (invoke를 통한 딕셔너리 전달)
result = tech_chain.invoke({"topic": topic})

print(result)

## 개요

Langchain은 자연어 처리를 위한 강력한 라이브러리로, 다양한 구성 요소를 통해 복잡한 작업을 간소화합니다. LLMChain은 이러한 Langchain의 구성 요소 중 하나로, 언어 모델(LLM: Language Model)을 활용하여 체인을 구성하는 데 사용됩니다. 이를 통해 다양한 자연어 처리 작업을 보다 효율적으로 수행할 수 있습니다.

## 핵심 개념

- **LLMChain**: LLMChain은 언어 모델을 기반으로 여러 단계의 처리 과정을 체인 형태로 연결합니다. 각 단계는 특정 작업을 수행하며, 이전 단계의 출력을 다음 단계의 입력으로 사용할 수 있습니다. 이를 통해 복잡한 작업을 작은 단위로 분리하여 처리할 수 있습니다.

- **언어 모델**: LLMChain에서 사용되는 언어 모델은 자연어를 이해하고 생성할 수 있는 인공지능 모델입니다. 일반적으로 GPT(Generative Pre-trained Transformer)와 같은 모델이 사용되며, 자연어 생성, 번역, 요약 등 다양한 작업에 활용됩니다.

- **체인 구성**: LLMChain은 여러 개의 모듈을 연결하여 체인을 구성합니다. 각 모듈은 특정한 기능을 수행하며, 체인 내의 데이터 흐름을 제어합니다. 이를 통해 복잡한 자연어 처리 파이프라인을 구축할 수 있습니다.

## 파이썬 코드 예제

```python
from langchain import LLMChain
from langchain.llms import OpenAI

# 언어 모델 인스턴스 생성
llm = OpenAI(api_key="your-api-key")

# LLMChain 생성
chain = LLMChain(llm=llm)

# 입력 데이터 정의
input_data = {
    "text": "Langchain의 LLMChain에 대해 설명해 주세요."
}

# 체인 실행
output = chain.run(input_data)

# 결과 출력
print(output)
```

이 예제에서는 Open

Gradio 웹 UI에서 ChatGPT처럼 대화 가능

- ChatGPT와 같은 대화형 어시스턴트를 LangChain으로 구현할 때는, 더 이상 과거의 ConversationBufferMemory와 LLMChain을 결합하는 복잡한 방식을 사용하지 않습니다.

최신 표준에서는
- RunnableWithMessageHistory를 사용하여 대화 맥락(Memory)을 체인에 주입하는 방식을 사용합니다. 이를 통해 코드의 복잡성을 대폭 낮추고, 어떤 모델이든 쉽게 교체할 수 있는 구조를 만듭니다.

In [17]:
!pip install -qU gradio

In [ ]:
# 최신 표준 리팩토링 코드

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
import os
from google.colab import userdata

# 1. API 키 설정
os.environ['OPENAI_API_KEY'] = userdata.get("OPENAI_API_KEY")

# 2. 모델 설정
model = ChatOpenAI(model = "gpt-4o", temperature= 0.7)

# 3. 최신 방식 :  ChatPromptTemplate의 리스트 결합및 구성
# 역할 기반 프롬프트 정의
prompt = ChatPromptTemplate.from_messages(
    [
        ("system",
         "너는 숙련된 기술 문서 작성자야."
         "개발자들을 위한 문서를 마크다운 형식으로 작성해."
         "구조는 다음과 같아 : \n\n"
         "## 개요\n\n"
         "## 핵심 개념\n\n"
         "## 파이썬 코드 예제\n\n"
         "## 참고 자료\n\n"
         "항상 한국어로 작성해."),
        ("human", "{topic}")
    ]
)

# 4. LCEL 파이프라인 구성
# prompt | model
tech_chain = prompt | model | StrOutputParser()


topic = "Langchain의 LLMChain 구성 요소 설명해줘"

# 5. 실행 (invoke를 통한 딕셔너리 전달)
result = tech_chain.invoke({"topic": topic})

print(result)

In [18]:
# 코드 상단 주석: 패키지 설치 명령어
# !pip install -q -U langchain-core langchain-openai langchain-community gradio

import os
from google.colab import userdata
import gradio as gr
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# 1. API 키 설정
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# 2. 모델 설정 (모델 이름만 바꾸면 됩니다: gpt-4o, gpt-4o-mini 등)
model = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# 3. 프롬프트 정의
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 사용자에게 친절하게 답변하는 AI 어시스턴트입니다."),
    MessagesPlaceholder(variable_name="history"), # 대화 맥락이 들어갈 자리
    ("human", "{question}")
])

# 4. 체인 구성
chain = prompt | model

# 5. 메모리 관리 설정
store = {} # 세션별 대화 저장을 위한 딕셔너리

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# 6. RunnableWithMessageHistory로 래핑
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history"
)

# 7. Gradio UI 구현
def respond(message, chat_history):
    response = chain_with_history.invoke(
        {"question": message},
        config={"configurable": {"session_id": "default_user"}}
    )
    return response.content

# Gradio 인터페이스 실행
gr.ChatInterface(fn=respond).launch(share=True)

/tmp/ipykernel_1763/854742738.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory
/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://51f426f12d417d4a37.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
